# OneNMT · HF preflight and separate candidate repairs

For Colab/Kaggle, or a local Jupyter kernel using the Titan V guide's environment. Read `docs/preflight.md` and `docs/repairs.md` first. This notebook does not train OneNMT or convert its architecture. Every repair writes a new candidate; initialized rows still need training and quality evaluation.

The repository tests use synthetic assets/tiny random HF models. This notebook has not been executed with full OneNMT weights or on hosted GPUs. Keep immutable originals and independent retained-language/adaptation probes.

In [ ]:
from pathlib import Path
import os, subprocess, sys
if Path('/kaggle/working').exists():
    ROOT = Path('/kaggle/working/BhaashikMTFinetuning')
elif Path('/content').exists():
    ROOT = Path('/content/BhaashikMTFinetuning')
else:
    ROOT = Path.cwd()  # Start Jupyter in the local checkout.
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/bhaashik/BhaashikMTFinetuning.git',str(ROOT)], check=True)
assert (ROOT/'pyproject.toml').exists(), 'Point ROOT to the checkout'
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[train,repairs]'], check=True)


## Supply local assets and reviewed probes

Original assets: `fairseq_dict.json`, `onemtv3b_spm.model`, config/generation config, and optional safetensors for header inspection (required for HF generation/resizing). Keep original and candidate paths separate. On Kaggle copy read-only input assets into a deliberate local original folder; on Colab back up candidates/reports to persistent storage.

Replace the illustrative probes for research. Include `src_lang`, `tgt_lang` and `group` (`retention` or `adaptation`) on each row.

In [ ]:
ORIGINAL = ROOT/'data/local/onenmt-original'
SOURCE = ORIGINAL  # Point to a previous candidate when applying another separate operation.
PROBES = ROOT/'data/preflight/probes.jsonl'  # Replace with reviewed real probes.
REPORTS = ROOT/'runs/onenmt-checks'
REPORTS.mkdir(parents=True, exist_ok=True)
HUB_REVISION = None  # Immutable Hub commit for an optional download.


In [ ]:
DOWNLOAD_ORIGINAL = False
if DOWNLOAD_ORIGINAL:
    assert HUB_REVISION, 'Pin the model revision before downloading'
    from huggingface_hub import snapshot_download
    snapshot_download('ltrciiith/bhashaverse', revision=HUB_REVISION, local_dir=str(ORIGINAL),
        allow_patterns=['fairseq_dict.json','onemtv3b_spm.model','config.json','generation_config.json',
                        '*.safetensors','model.safetensors.index.json','README.md','LICENSE*'])
print('Original folder:', ORIGINAL)


## Diagnose first

This is read-only. A blocked report identifies a prerequisite or incompatibility; do not automatically apply every possible repair. Vocabulary/dictionary mutation currently requires identical source/target dictionaries for the shared HF MBart path.

In [ ]:
from bhaashik.preflight import preflight
original_report = preflight(ORIGINAL, PROBES, REPORTS/'original.json')
print(original_report['structural_status'])
print('Blocking:', original_report['blocking_findings'])
print('Review:', original_report['review_findings'])


## Select exactly one operation

Default `ACTION='none'` does nothing. Set input paths and a new destination deliberately. Rerunning into an existing output is refused. For a staged extension: spm-extend → append → resize, updating SOURCE to the previous stage and ACTION/destination each time. You can skip tokenizer/dictionary changes when coverage is adequate.

Dictionary restoration uses SOURCE as the trusted original pair; it never imports a model trained under reordered IDs. SPM retraining is a high-risk ablation. Resizing loads models on CPU and verifies save/reload, so it requires substantial host RAM/disk.

In [ ]:
ACTION = 'none'  # generation, dictionary, append, spm-extend, spm-train, spm-replace, resize
TOKENS = ROOT/'data/preflight/selected-tokens.example.json'
PROPOSED_DICTIONARY = ROOT/'data/local/proposed/fairseq_dict.json'
TRAINING_TEXT = ROOT/'data/local/mixed-old-new.txt'
REPLACEMENT_SPM = ROOT/'data/local/external/tokenizer.model'
SPM_VOCAB_SIZE, SPM_TYPE = 64000, 'unigram'  # Explicit ablation choices.
DESTINATION = ROOT/'data/local'/f'candidate-{ACTION}'
CANDIDATE = None
if ACTION != 'none':
    from bhaashik.repairs.generation import repair_generation
    from bhaashik.repairs.dictionary import repair_dictionary, repair_append
    from bhaashik.repairs.sentencepiece import repair_spm_extend, repair_spm_train, repair_spm_replace
    from bhaashik.repairs.embeddings import repair_resize
    operations = {
        'generation': lambda: repair_generation(SOURCE, DESTINATION),
        'dictionary': lambda: repair_dictionary(SOURCE, PROPOSED_DICTIONARY, DESTINATION),
        'append': lambda: repair_append(SOURCE, DESTINATION, tokens=TOKENS),
        'spm-extend': lambda: repair_spm_extend(SOURCE, TOKENS, DESTINATION),
        'spm-train': lambda: repair_spm_train(SOURCE, TRAINING_TEXT, DESTINATION, SPM_VOCAB_SIZE, SPM_TYPE),
        'spm-replace': lambda: repair_spm_replace(SOURCE, REPLACEMENT_SPM, DESTINATION),
        'resize': lambda: repair_resize(SOURCE, ORIGINAL, DESTINATION),
    }
    assert ACTION in operations, 'Choose a listed operation'
    manifest = operations[ACTION]()
    CANDIDATE = DESTINATION
    print(manifest['status'], 'Embedding training pending:', manifest['requires_embedding_training'])
else:
    print('No repair selected; original assets unchanged.')


In [ ]:
if CANDIDATE:
    candidate_report = preflight(ORIGINAL, PROBES, REPORTS/f'{ACTION}-preflight.json', candidate_assets=CANDIDATE)
    print(candidate_report['structural_status'])
    print(candidate_report['blocking_findings'], candidate_report['review_findings'])
else:
    print('No candidate created. Set CANDIDATE to an existing reviewed folder if comparing an earlier stage.')


## Optional HF behavior check

Requires complete local safetensors. Run only after resolving structural blockers. Standard HF MBart only, custom Fairseq piece/ID codec, no AutoTokenizer or architecture reimplementation. FP16 requires CUDA; choose CPU/FP32 if sufficient host RAM is available. Independent reference predictions must use the same probes/assets/greedy decoding settings. Without them, parity remains unverified.

In [ ]:
RUN_HF_REGRESSION = False
REFERENCE_PREDICTIONS = None  # Optional trusted JSONL; do not fabricate it.
if RUN_HF_REGRESSION:
    assert CANDIDATE and CANDIDATE.exists(), 'Supply a candidate'
    import torch
    from bhaashik.hf_regression import hf_regression
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    result = hf_regression(ORIGINAL, CANDIDATE, PROBES, REPORTS/f'{ACTION}-behavior.json',
        reference=REFERENCE_PREDICTIONS, device=device, precision='fp16' if device=='cuda' else 'fp32',
        max_new_tokens=64, batch_size=1)
    print(result['status'])
    print(result.get('blocking_findings', []), result.get('review_findings', []))
else:
    print('HF generation skipped; structural checks alone do not prove translation quality.')


## Export reports; preserve large assets separately

The ZIP below contains reports, not original/candidate model weights. Copy the chosen candidate and its repair manifest to persistent storage separately. Colab `/content` and Kaggle interactive files are ephemeral. Never publish private probes or incompatible model/data artifacts accidentally.

In [ ]:
import shutil
archive = shutil.make_archive(str(ROOT.parent/'onenmt-check-reports'), 'zip', root_dir=str(REPORTS))
print('Report archive:', archive)
if CANDIDATE:
    print('Back up candidate separately:', CANDIDATE)
